# Proyecto A: Sistema de Predicción de Riesgo Crediticio

## Preprocesamiento de datos

Este notebook prepara el conjunto German Credit para el entrenamiento de los
modelos de redes neuronales artificiales.

El proceso incluye:

- Carga del dataset original.
- Separación de variables numéricas y categóricas.
- Transformación de la variable objetivo.
- Conversión de variables categóricas mediante `get_dummies`.
- División de los datos en entrenamiento y prueba.
- Normalización mediante `MinMaxScaler`.
- Guardado de los datos procesados.

# 1. Importación de librerías

In [1]:
# Librerías para trabajar con los datos

import pandas as pd
import numpy as np

# Librerías para dividir y normalizar los datos

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Librería para guardar el escalador y otros objetos

import joblib

# 2. Importación de la fuente de datos

Se utiliza el archivo original `german.data`, que contiene 20 variables de
entrada y una variable objetivo.

La variable objetivo original utiliza la siguiente codificación:

- `1`: crédito bueno.
- `2`: crédito malo.

In [2]:
# Nombres de las columnas según la documentación oficial del dataset

columnas = [
    "estado_cuenta",
    "duracion_meses",
    "historial_crediticio",
    "proposito",
    "monto_credito",
    "cuenta_ahorros",
    "antiguedad_empleo",
    "tasa_cuotas",
    "estado_personal",
    "otros_deudores",
    "tiempo_residencia",
    "propiedad",
    "edad",
    "otros_planes_pago",
    "vivienda",
    "creditos_existentes",
    "trabajo",
    "personas_dependientes",
    "telefono",
    "trabajador_extranjero",
    "clase"
]

In [3]:
# Lectura del dataset original

df = pd.read_csv(
    "../data/raw/german.data",
    sep=r"\s+",
    header=None,
    names=columnas
)

df.head()

,estado_cuenta,duracion_meses,historial_crediticio,proposito,monto_credito,cuenta_ahorros,antiguedad_empleo,tasa_cuotas,estado_personal,otros_deudores,...,propiedad,edad,otros_planes_pago,vivienda,creditos_existentes,trabajo,personas_dependientes,telefono,trabajador_extranjero,clase
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


# 3. Verificación inicial

Antes de realizar las transformaciones, se comprueba nuevamente la estructura
y calidad general de los datos.

In [4]:
# Resumen inicial del dataset

print("Cantidad de registros:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])
print("Valores nulos:", df.isnull().sum().sum())
print("Registros duplicados:", df.duplicated().sum())

Cantidad de registros: 1000
Cantidad de columnas: 21
Valores nulos: 0
Registros duplicados: 0


# 4. Separación de variables numéricas y categóricas

Las variables numéricas serán normalizadas utilizando `MinMaxScaler`.

Las variables categóricas serán convertidas a columnas numéricas mediante
`get_dummies`.

In [5]:
# Variables numéricas

columnas_numericas = [
    "duracion_meses",
    "monto_credito",
    "tasa_cuotas",
    "tiempo_residencia",
    "edad",
    "creditos_existentes",
    "personas_dependientes"
]

In [6]:
# Variables categóricas

columnas_categoricas = [
    "estado_cuenta",
    "historial_crediticio",
    "proposito",
    "cuenta_ahorros",
    "antiguedad_empleo",
    "estado_personal",
    "otros_deudores",
    "propiedad",
    "otros_planes_pago",
    "vivienda",
    "trabajo",
    "telefono",
    "trabajador_extranjero"
]

In [7]:
# Comprobación de la cantidad de variables

print("Variables numéricas:", len(columnas_numericas))
print("Variables categóricas:", len(columnas_categoricas))
print("Total de variables de entrada:",
      len(columnas_numericas) + len(columnas_categoricas))

Variables numéricas: 7
Variables categóricas: 13
Total de variables de entrada: 20


# 5. Transformación de la variable objetivo binaria

La variable `clase` utiliza originalmente la siguiente codificación:

- `1`: crédito bueno.
- `2`: crédito malo.

Para facilitar el entrenamiento del modelo de clasificación binaria, se
transformará a:

- `0`: crédito bueno.
- `1`: crédito malo.

Esta transformación no modifica la clasificación original de los registros.
Únicamente cambia su representación numérica.

In [8]:
# Distribución original de la variable objetivo
df["clase"].value_counts().sort_index()

clase
1    700
2    300
Name: count, dtype: int64

In [9]:
# Transformación de la clase original

df["riesgo_binario"] = df["clase"].map({
    1: 0,
    2: 1
})

In [10]:
# Comparación entre la clase original y la variable binaria
df[["clase", "riesgo_binario"]].head(10)

,clase,riesgo_binario
0,1,0
1,2,1
2,1,0
3,1,0
4,2,1
5,1,0
6,1,0
7,1,0
8,1,0
9,2,1


In [11]:
# Distribución de la variable objetivo transformada
df["riesgo_binario"].value_counts().sort_index()

riesgo_binario
0    700
1    300
Name: count, dtype: int64

In [12]:
# Verificación de la transformación

print("Valores nulos en riesgo_binario:",
      df["riesgo_binario"].isnull().sum())

print("Cantidad de créditos buenos:",
      (df["riesgo_binario"] == 0).sum())

print("Cantidad de créditos malos:",
      (df["riesgo_binario"] == 1).sum())

Valores nulos en riesgo_binario: 0
Cantidad de créditos buenos: 700
Cantidad de créditos malos: 300


### Resultado de la transformación binaria

La variable objetivo se transformó correctamente:

- 700 registros tienen el valor `0`, correspondiente a créditos buenos.
- 300 registros tienen el valor `1`, correspondiente a créditos malos.
- No se generaron valores nulos durante la transformación.

Se utiliza el valor `1` para representar el crédito malo, de manera que la
probabilidad obtenida por el modelo binario pueda interpretarse como la
probabilidad estimada de riesgo crediticio malo.

# 6. Separación de las variables de entrada y salida

Las variables de entrada se almacenan en `X`, mientras que la variable objetivo
binaria se almacena en `y`.

La columna original `clase` no se incluirá entre las entradas, porque contiene
la respuesta que se desea predecir.

In [13]:
# Variables de entrada

X = df.drop(
    columns=["clase", "riesgo_binario"]
)

# Variable objetivo binaria

y = df["riesgo_binario"]

In [14]:
# Dimensiones de las variables de entrada y salida

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Dimensiones de X: (1000, 20)
Dimensiones de y: (1000,)


In [15]:
# Comprobación de las columnas excluidas

print("¿clase está dentro de X?", "clase" in X.columns)
print("¿riesgo_binario está dentro de X?", "riesgo_binario" in X.columns)

¿clase está dentro de X? False
¿riesgo_binario está dentro de X? False


# 7. Conversión de variables categóricas

Las redes neuronales trabajan con valores numéricos. Por esta razón, las
variables categóricas se transformarán mediante `get_dummies`.

Este procedimiento crea columnas separadas con valores `0` y `1` para
representar cada categoría.

Las variables numéricas conservarán sus columnas originales.

In [16]:
# Tipos de datos antes de aplicar get_dummies

X.dtypes.value_counts()

str      13
int64     7
Name: count, dtype: int64

In [17]:
# Conversión de las variables categóricas a variables dummy

X_transformado = pd.get_dummies(
    X,
    columns=columnas_categoricas,
    dtype=int
)

In [18]:
# Visualización de los datos transformados
X_transformado.head()

,duracion_meses,monto_credito,tasa_cuotas,tiempo_residencia,edad,creditos_existentes,personas_dependientes,estado_cuenta_A11,estado_cuenta_A12,estado_cuenta_A13,...,vivienda_A152,vivienda_A153,trabajo_A171,trabajo_A172,trabajo_A173,trabajo_A174,telefono_A191,telefono_A192,trabajador_extranjero_A201,trabajador_extranjero_A202
0,6,1169,4,4,67,2,1,1,0,0,...,1,0,0,0,1,0,0,1,1,0
1,48,5951,2,2,22,1,1,0,1,0,...,1,0,0,0,1,0,1,0,1,0
2,12,2096,2,3,49,1,2,0,0,0,...,1,0,0,1,0,0,1,0,1,0
3,42,7882,2,4,45,1,2,1,0,0,...,0,1,0,0,1,0,1,0,1,0
4,24,4870,3,4,53,2,2,1,0,0,...,0,1,0,0,1,0,1,0,1,0


In [19]:
# Comparación antes y después de get_dummies
print("Dimensiones originales de X:", X.shape)
print("Dimensiones después de get_dummies:", X_transformado.shape)

Dimensiones originales de X: (1000, 20)
Dimensiones después de get_dummies: (1000, 61)


In [20]:
# Comprobación de los tipos de datos después de la transformación
X_transformado.dtypes.value_counts()

int64    61
Name: count, dtype: int64

In [21]:
# Cantidad de columnas no numéricas después de get_dummies

columnas_no_numericas = X_transformado.select_dtypes(
    exclude=np.number
).columns

print("Cantidad de columnas no numéricas:",
      len(columnas_no_numericas))

Cantidad de columnas no numéricas: 0


In [22]:
# Verificación de valores nulos después de get_dummies
print(
    "Valores nulos después de get_dummies:",
    X_transformado.isnull().sum().sum()
)

Valores nulos después de get_dummies: 0


In [23]:
# Lista de columnas finales utilizadas por los modelos

columnas_modelo = X_transformado.columns.tolist()

print("Cantidad de columnas para los modelos:",
      len(columnas_modelo))

columnas_modelo[:10]

Cantidad de columnas para los modelos: 61


['duracion_meses',
 'monto_credito',
 'tasa_cuotas',
 'tiempo_residencia',
 'edad',
 'creditos_existentes',
 'personas_dependientes',
 'estado_cuenta_A11',
 'estado_cuenta_A12',
 'estado_cuenta_A13']

### Resultado de la conversión categórica

Las 13 variables categóricas fueron transformadas correctamente mediante
`get_dummies`.

Después de la transformación, el conjunto de entrada contiene `61` columnas
numéricas. No permanecen columnas de tipo texto y no se generaron valores
nulos.

La lista y el orden de estas columnas deberán conservarse para utilizarlos
durante el entrenamiento de los modelos, la API y la interfaz.

# 8. División de los datos en entrenamiento y prueba

El conjunto de datos se divide en:

- 80 % para entrenamiento.
- 20 % para prueba.

Se utiliza `stratify=y` para conservar aproximadamente la proporción original
de créditos buenos y malos en ambos conjuntos.

El conjunto de prueba se reservará para la evaluación final de los modelos.
Durante el entrenamiento, los compañeros utilizarán `validation_split` sobre
el conjunto de entrenamiento.

In [24]:
# División de los datos en entrenamiento y prueba

X_train, X_test, y_train, y_test = train_test_split(
    X_transformado,
    y,
    test_size=0.20,
    random_state=101,
    stratify=y
)

In [25]:
# Dimensiones de los conjuntos

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 61)
X_test: (200, 61)
y_train: (800,)
y_test: (200,)


In [26]:
# Distribución de clases en entrenamiento

print("Distribución en entrenamiento:")
print(y_train.value_counts().sort_index())

print("\nPorcentajes en entrenamiento:")
print(
    (y_train.value_counts(normalize=True).sort_index() * 100).round(2)
)

Distribución en entrenamiento:
riesgo_binario
0    560
1    240
Name: count, dtype: int64

Porcentajes en entrenamiento:
riesgo_binario
0    70.0
1    30.0
Name: proportion, dtype: float64


In [27]:
# Distribución de clases en prueba

print("Distribución en prueba:")
print(y_test.value_counts().sort_index())

print("\nPorcentajes en prueba:")
print(
    (y_test.value_counts(normalize=True).sort_index() * 100).round(2)
)

Distribución en prueba:
riesgo_binario
0    140
1     60
Name: count, dtype: int64

Porcentajes en prueba:
riesgo_binario
0    70.0
1    30.0
Name: proportion, dtype: float64


### Resultado de la división

El conjunto de entrenamiento contiene 800 registros y el conjunto de prueba
contiene 200 registros. Ambos tienen 61 variables de entrada.

La proporción original se conservó en los dos conjuntos:

- 70 % de créditos buenos.
- 30 % de créditos malos.

El conjunto de prueba permanecerá separado y se utilizará únicamente para la
evaluación final de los modelos.

# 9. Normalización de las variables

Las variables numéricas presentan escalas diferentes. Por ejemplo, la edad
se expresa en años y el monto del crédito puede alcanzar valores considerablemente
mayores.

Se utiliza `MinMaxScaler` para transformar las entradas a una escala entre
0 y 1.

El escalador se ajusta únicamente con el conjunto de entrenamiento para evitar
que la información del conjunto de prueba influya en la preparación del modelo.

In [28]:
# Creación del escalador

scaler = MinMaxScaler()

# El escalador aprende únicamente de los datos de entrenamiento

X_train_scaled = scaler.fit_transform(X_train)

# El conjunto de prueba se transforma con el mismo escalador

X_test_scaled = scaler.transform(X_test)

In [29]:
# Conversión de los datos escalados nuevamente a DataFrame

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=columnas_modelo,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=columnas_modelo,
    index=X_test.index
)

In [30]:
# Dimensiones después de la normalización

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (800, 61)
X_test_scaled: (200, 61)


In [31]:
# Primeros registros normalizados
X_train_scaled.head()

,duracion_meses,monto_credito,tasa_cuotas,tiempo_residencia,edad,creditos_existentes,personas_dependientes,estado_cuenta_A11,estado_cuenta_A12,estado_cuenta_A13,...,vivienda_A152,vivienda_A153,trabajo_A171,trabajo_A172,trabajo_A173,trabajo_A174,telefono_A191,telefono_A192,trabajador_extranjero_A201,trabajador_extranjero_A202
76,0.558824,0.203273,1.000000,0.666667,0.267857,0.000000,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
573,0.161765,0.029204,1.000000,1.000000,0.053571,0.000000,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
519,0.029412,0.023363,1.000000,1.000000,0.303571,0.333333,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
911,0.294118,0.245757,0.333333,1.000000,0.107143,0.000000,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
310,0.647059,0.281298,0.666667,1.000000,0.375000,0.000000,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [32]:
# Verificación del rango de los datos de entrenamiento

print("Valor mínimo en entrenamiento:",
      X_train_scaled.min().min())

print("Valor máximo en entrenamiento:",
      X_train_scaled.max().max())

Valor mínimo en entrenamiento: 0.0
Valor máximo en entrenamiento: 1.0


In [33]:
# Verificación de valores nulos después de normalizar

print(
    "Valores nulos en entrenamiento:",
    X_train_scaled.isnull().sum().sum()
)

print(
    "Valores nulos en prueba:",
    X_test_scaled.isnull().sum().sum()
)

Valores nulos en entrenamiento: 0
Valores nulos en prueba: 0


### Resultado de la normalización

El escalador fue ajustado únicamente con los 800 registros del conjunto de
entrenamiento.

Posteriormente, el mismo escalador se utilizó para transformar los conjuntos
de entrenamiento y prueba. Ambos conservan las 61 columnas y no contienen
valores nulos.

Los datos de entrenamiento se encuentran en una escala entre 0 y 1, por lo
que están preparados para ser utilizados por las redes neuronales.

# 10. Guardado de los datos procesados

Se guardan los conjuntos de entrenamiento y prueba correspondientes al modelo
binario.

También se guardan el escalador, la lista ordenada de columnas y la información
necesaria para repetir el mismo preprocesamiento con datos nuevos.

Estos archivos serán utilizados posteriormente durante el entrenamiento,
la evaluación, la API y la interfaz.

In [34]:
# Preparación de las variables objetivo para guardar

y_train_guardar = y_train.to_frame(
    name="riesgo_binario"
)

y_test_guardar = y_test.to_frame(
    name="riesgo_binario"
)

In [35]:
# Comprobación de los índices antes de guardar

print(
    "Índices de entrenamiento coinciden:",
    X_train_scaled.index.equals(y_train_guardar.index)
)

print(
    "Índices de prueba coinciden:",
    X_test_scaled.index.equals(y_test_guardar.index)
)

Índices de entrenamiento coinciden: True
Índices de prueba coinciden: True


In [36]:
from pathlib import Path

In [37]:
# Definición de las carpetas de salida

carpeta_datos = Path("../data/processed")
carpeta_modelos = Path("../models")

# Creación de las carpetas si no existen

carpeta_datos.mkdir(
    parents=True,
    exist_ok=True
)

carpeta_modelos.mkdir(
    parents=True,
    exist_ok=True
)

In [38]:
# Guardado de los datos del modelo binario

X_train_scaled.to_csv(
    carpeta_datos / "X_train_binario.csv",
    index=False
)

X_test_scaled.to_csv(
    carpeta_datos / "X_test_binario.csv",
    index=False
)

y_train_guardar.to_csv(
    carpeta_datos / "y_train_binario.csv",
    index=False
)

y_test_guardar.to_csv(
    carpeta_datos / "y_test_binario.csv",
    index=False
)

In [39]:
# Guardado del escalador

joblib.dump(
    scaler,
    carpeta_modelos / "scaler.pkl"
)

['..\\models\\scaler.pkl']

In [40]:
# Guardado del orden de las columnas

joblib.dump(
    columnas_modelo,
    carpeta_modelos / "columnas_modelo.pkl"
)

['..\\models\\columnas_modelo.pkl']

In [41]:
# Información necesaria para repetir el preprocesamiento

informacion_preprocesamiento = {
    "columnas_originales": X.columns.tolist(),
    "columnas_numericas": columnas_numericas,
    "columnas_categoricas": columnas_categoricas,
    "columnas_modelo": columnas_modelo,
    "clase_original": {
        1: "Bueno",
        2: "Malo"
    },
    "clase_binaria": {
        0: "Bueno",
        1: "Malo"
    }
}

joblib.dump(
    informacion_preprocesamiento,
    carpeta_modelos / "informacion_preprocesamiento.pkl"
)

['..\\models\\informacion_preprocesamiento.pkl']

In [42]:
# Archivos generados para el modelo binario

archivos_generados = [
    carpeta_datos / "X_train_binario.csv",
    carpeta_datos / "X_test_binario.csv",
    carpeta_datos / "y_train_binario.csv",
    carpeta_datos / "y_test_binario.csv",
    carpeta_modelos / "scaler.pkl",
    carpeta_modelos / "columnas_modelo.pkl",
    carpeta_modelos / "informacion_preprocesamiento.pkl"
]

for archivo in archivos_generados:
    print(
        archivo.name,
        "->",
        "Creado" if archivo.exists() else "No creado"
    )

X_train_binario.csv -> Creado
X_test_binario.csv -> Creado
y_train_binario.csv -> Creado
y_test_binario.csv -> Creado
scaler.pkl -> Creado
columnas_modelo.pkl -> Creado
informacion_preprocesamiento.pkl -> Creado


# 12. Verificar que los archivos fueron creados

In [43]:
# Comprobación de los archivos guardados

X_train_comprobacion = pd.read_csv(
    carpeta_datos / "X_train_binario.csv"
)

X_test_comprobacion = pd.read_csv(
    carpeta_datos / "X_test_binario.csv"
)

y_train_comprobacion = pd.read_csv(
    carpeta_datos / "y_train_binario.csv"
)

y_test_comprobacion = pd.read_csv(
    carpeta_datos / "y_test_binario.csv"
)

print("X_train guardado:", X_train_comprobacion.shape)
print("X_test guardado:", X_test_comprobacion.shape)
print("y_train guardado:", y_train_comprobacion.shape)
print("y_test guardado:", y_test_comprobacion.shape)

X_train guardado: (800, 61)
X_test guardado: (200, 61)
y_train guardado: (800, 1)
y_test guardado: (200, 1)


### Archivos preparados para el modelo binario

Se generaron correctamente los conjuntos de entrenamiento y prueba:

- `X_train_binario.csv`: 800 registros y 61 entradas.
- `X_test_binario.csv`: 200 registros y 61 entradas.
- `y_train_binario.csv`: 800 resultados.
- `y_test_binario.csv`: 200 resultados.

También se guardaron:

- `scaler.pkl`: escalador ajustado únicamente con entrenamiento.
- `columnas_modelo.pkl`: columnas y orden exacto del modelo.
- `informacion_preprocesamiento.pkl`: información para repetir las
  transformaciones con datos nuevos.

La codificación de la salida binaria es:

- `0`: crédito bueno.
- `1`: crédito malo.

# 13. Creación de la variable objetivo multiclase

El dataset original contiene únicamente dos clasificaciones:

- Crédito bueno.
- Crédito malo.

Para construir el segundo modelo solicitado, se crearán cuatro niveles:

- `0`: Bajo.
- `1`: Medio.
- `2`: Alto.
- `3`: Crítico.

Estos niveles no son etiquetas originales del dataset. Corresponden a una
clasificación académica construida de manera transparente y reproducible.

La clase original se utiliza como base:

- Los créditos buenos se dividen entre riesgo Bajo y Medio.
- Los créditos malos se dividen entre riesgo Alto y Crítico.

Para diferenciar los niveles dentro de cada grupo se utiliza un puntaje de
exposición basado en la duración, monto, estado de cuenta, ahorros e historial
crediticio.

Los umbrales numéricos se calculan únicamente con el conjunto de entrenamiento
para evitar que la información del conjunto de prueba influya en su definición.

# 14. Recuperar los registros originales de entrenamiento y prueba

In [44]:
# Registros originales correspondientes a entrenamiento y prueba

df_train_original = df.loc[X_train.index].copy()
df_test_original = df.loc[X_test.index].copy()

print("Registros originales de entrenamiento:",
      df_train_original.shape)

print("Registros originales de prueba:",
      df_test_original.shape)

Registros originales de entrenamiento: (800, 22)
Registros originales de prueba: (200, 22)


# 15. Calcular umbrales usando entrenamiento

In [45]:
# Umbrales calculados únicamente con entrenamiento

mediana_duracion = df_train_original["duracion_meses"].median()
mediana_monto = df_train_original["monto_credito"].median()

print("Mediana de duración:",
      mediana_duracion)

print("Mediana del monto:",
      mediana_monto)


Mediana de duración: 18.0
Mediana del monto: 2302.5


# 16. Función para calcular el puntaje

In [46]:
# Función para calcular el puntaje de exposición

def calcular_puntaje_exposicion(
    fila,
    limite_duracion,
    limite_monto
):
    puntaje = 0

    # Duración superior a la mediana de entrenamiento
    if fila["duracion_meses"] > limite_duracion:
        puntaje += 1

    # Monto superior a la mediana de entrenamiento
    if fila["monto_credito"] > limite_monto:
        puntaje += 1

    # Estados de cuenta con mayor proporción observada de créditos malos
    if fila["estado_cuenta"] in ("A11", "A12"):
        puntaje += 1

    # Categorías de ahorro con mayor proporción observada de créditos malos
    if fila["cuenta_ahorros"] in ("A61", "A62"):
        puntaje += 1

    # Historiales con mayor proporción observada de créditos malos
    if fila["historial_crediticio"] in ("A30", "A31"):
        puntaje += 1

    return puntaje

# 17. Aplicar el puntaje

In [47]:
# Puntaje de exposición para entrenamiento

df_train_original["puntaje_exposicion"] = (
    df_train_original.apply(
        calcular_puntaje_exposicion,
        axis=1,
        limite_duracion=mediana_duracion,
        limite_monto=mediana_monto
    )
)

# Puntaje de exposición para prueba

df_test_original["puntaje_exposicion"] = (
    df_test_original.apply(
        calcular_puntaje_exposicion,
        axis=1,
        limite_duracion=mediana_duracion,
        limite_monto=mediana_monto
    )
)


In [48]:
print("Puntajes en entrenamiento:")
print(
    df_train_original["puntaje_exposicion"]
    .value_counts()
    .sort_index()
)

print("\nPuntajes en prueba:")
print(
    df_test_original["puntaje_exposicion"]
    .value_counts()
    .sort_index()
)

Puntajes en entrenamiento:
puntaje_exposicion
0     50
1    164
2    251
3    192
4    123
5     20
Name: count, dtype: int64

Puntajes en prueba:
puntaje_exposicion
0    12
1    46
2    59
3    42
4    34
5     7
Name: count, dtype: int64


In [49]:
print(
    "Puntaje mínimo:",
    df_train_original["puntaje_exposicion"].min()
)

print(
    "Puntaje máximo:",
    df_train_original["puntaje_exposicion"].max()
)

Puntaje mínimo: 0
Puntaje máximo: 5


# 18. Definir Bajo, Medio, Alto y Crítico

In [50]:
# Umbral para dividir créditos buenos

umbral_buenos = df_train_original.loc[
    df_train_original["riesgo_binario"] == 0,
    "puntaje_exposicion"
].median()

# Umbral para dividir créditos malos

umbral_malos = df_train_original.loc[
    df_train_original["riesgo_binario"] == 1,
    "puntaje_exposicion"
].median()

print("Umbral para créditos buenos:",
      umbral_buenos)

print("Umbral para créditos malos:",
      umbral_malos)

Umbral para créditos buenos: 2.0
Umbral para créditos malos: 3.0


# 19. Función para asignar el nivel

In [51]:
# Función para crear los cuatro niveles de riesgo

def asignar_nivel_riesgo(
    riesgo_binario,
    puntaje,
    limite_buenos,
    limite_malos
):
    if riesgo_binario == 0:
        if puntaje <= limite_buenos:
            return 0  # Bajo
        else:
            return 1  # Medio

    else:
        if puntaje <= limite_malos:
            return 2  # Alto
        else:
            return 3  # Crítico

In [52]:
# Niveles de riesgo para entrenamiento

y_train_multiclase = df_train_original.apply(
    lambda fila: asignar_nivel_riesgo(
        fila["riesgo_binario"],
        fila["puntaje_exposicion"],
        umbral_buenos,
        umbral_malos
    ),
    axis=1
)

y_train_multiclase.name = "nivel_riesgo"

In [53]:
# Niveles de riesgo para prueba

y_test_multiclase = df_test_original.apply(
    lambda fila: asignar_nivel_riesgo(
        fila["riesgo_binario"],
        fila["puntaje_exposicion"],
        umbral_buenos,
        umbral_malos
    ),
    axis=1
)

y_test_multiclase.name = "nivel_riesgo"

# 20. Comprobar los niveles

In [54]:
# Significado de las clases multiclase

nombres_niveles = {
    0: "Bajo",
    1: "Medio",
    2: "Alto",
    3: "Crítico"
}

print("Distribución en entrenamiento:")
print(
    y_train_multiclase
    .value_counts()
    .sort_index()
    .rename(index=nombres_niveles)
)

print("\nDistribución en prueba:")
print(
    y_test_multiclase
    .value_counts()
    .sort_index()
    .rename(index=nombres_niveles)
)

Distribución en entrenamiento:
nivel_riesgo
Bajo       372
Medio      188
Alto       163
Crítico     77
Name: count, dtype: int64

Distribución en prueba:
nivel_riesgo
Bajo       101
Medio       39
Alto        34
Crítico     26
Name: count, dtype: int64


In [55]:
print(
    "Valores nulos en entrenamiento:",
    y_train_multiclase.isnull().sum()
)

print(
    "Valores nulos en prueba:",
    y_test_multiclase.isnull().sum()
)

print(
    "Niveles encontrados:",
    sorted(y_train_multiclase.unique())
)

Valores nulos en entrenamiento: 0
Valores nulos en prueba: 0
Niveles encontrados: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


# 21. Comprobar la coherencia

In [56]:
# Tabla de coherencia entre riesgo binario y multiclase

tabla_coherencia = pd.crosstab(
    y_train,
    y_train_multiclase
)

tabla_coherencia.columns = [
    nombres_niveles[columna]
    for columna in tabla_coherencia.columns
]

tabla_coherencia.index = [
    "Bueno",
    "Malo"
]

tabla_coherencia

,Bajo,Medio,Alto,Crítico
Bueno,372,188,0,0
Malo,0,0,163,77


### Justificación de los cuatro niveles

Los niveles de riesgo se construyeron a partir de la clasificación original
y de un puntaje de exposición.

Los registros originalmente clasificados como buenos se dividieron entre
riesgo Bajo y Medio. Los registros originalmente clasificados como malos se
dividieron entre riesgo Alto y Crítico.

El puntaje considera duración, monto, estado de cuenta, condición de ahorros
e historial crediticio. Los umbrales numéricos y de puntaje fueron calculados
únicamente con el conjunto de entrenamiento.

Esta clasificación es una propuesta académica reproducible para el proyecto.
No corresponde a una escala oficial bancaria ni debe interpretarse como una
decisión crediticia para casos reales.

# 22. Guardar la salida multiclase

In [57]:
# Guardado de las variables objetivo multiclase

y_train_multiclase.to_frame().to_csv(
    carpeta_datos / "y_train_multiclase.csv",
    index=False
)

y_test_multiclase.to_frame().to_csv(
    carpeta_datos / "y_test_multiclase.csv",
    index=False
)

In [58]:
# Copias de las entradas para identificar el modelo multiclase

X_train_scaled.to_csv(
    carpeta_datos / "X_train_multiclase.csv",
    index=False
)

X_test_scaled.to_csv(
    carpeta_datos / "X_test_multiclase.csv",
    index=False
)

In [59]:
# Información para reproducir la clasificación multiclase

regla_multiclase = {
    "mediana_duracion": mediana_duracion,
    "mediana_monto": mediana_monto,
    "umbral_buenos": umbral_buenos,
    "umbral_malos": umbral_malos,
    "niveles": nombres_niveles,
    "estado_cuenta_riesgo": ["A11", "A12"],
    "cuenta_ahorros_riesgo": ["A61", "A62"],
    "historial_riesgo": ["A30", "A31"]
}

joblib.dump(
    regla_multiclase,
    carpeta_modelos / "regla_multiclase.pkl"
)

['..\\models\\regla_multiclase.pkl']

# 23. Verificación final de los archivos generados

Se comprueba que todos los archivos necesarios para los modelos binario y
multiclase fueron creados correctamente.

Los dos modelos utilizarán las mismas 61 columnas de entrada, pero tendrán
variables objetivo diferentes.

In [60]:
# Lista final de archivos generados

archivos_finales = [
    carpeta_datos / "X_train_binario.csv",
    carpeta_datos / "X_test_binario.csv",
    carpeta_datos / "y_train_binario.csv",
    carpeta_datos / "y_test_binario.csv",
    carpeta_datos / "X_train_multiclase.csv",
    carpeta_datos / "X_test_multiclase.csv",
    carpeta_datos / "y_train_multiclase.csv",
    carpeta_datos / "y_test_multiclase.csv",
    carpeta_modelos / "scaler.pkl",
    carpeta_modelos / "columnas_modelo.pkl",
    carpeta_modelos / "informacion_preprocesamiento.pkl",
    carpeta_modelos / "regla_multiclase.pkl"
]

for archivo in archivos_finales:
    estado = "Creado" if archivo.exists() else "No creado"
    print(f"{archivo.name}: {estado}")

X_train_binario.csv: Creado
X_test_binario.csv: Creado
y_train_binario.csv: Creado
y_test_binario.csv: Creado
X_train_multiclase.csv: Creado
X_test_multiclase.csv: Creado
y_train_multiclase.csv: Creado
y_test_multiclase.csv: Creado
scaler.pkl: Creado
columnas_modelo.pkl: Creado
informacion_preprocesamiento.pkl: Creado
regla_multiclase.pkl: Creado


In [61]:
# Comprobación de las dimensiones de los archivos procesados

archivos_csv = {
    "X_train_binario": carpeta_datos / "X_train_binario.csv",
    "X_test_binario": carpeta_datos / "X_test_binario.csv",
    "y_train_binario": carpeta_datos / "y_train_binario.csv",
    "y_test_binario": carpeta_datos / "y_test_binario.csv",
    "X_train_multiclase": carpeta_datos / "X_train_multiclase.csv",
    "X_test_multiclase": carpeta_datos / "X_test_multiclase.csv",
    "y_train_multiclase": carpeta_datos / "y_train_multiclase.csv",
    "y_test_multiclase": carpeta_datos / "y_test_multiclase.csv"
}

for nombre, ruta in archivos_csv.items():
    datos_comprobacion = pd.read_csv(ruta)
    print(nombre, datos_comprobacion.shape)

X_train_binario (800, 61)
X_test_binario (200, 61)
y_train_binario (800, 1)
y_test_binario (200, 1)
X_train_multiclase (800, 61)
X_test_multiclase (200, 61)
y_train_multiclase (800, 1)
y_test_multiclase (200, 1)


In [62]:
# Comprobación de las clases almacenadas

y_binario_comprobacion = pd.read_csv(
    carpeta_datos / "y_train_binario.csv"
)

y_multiclase_comprobacion = pd.read_csv(
    carpeta_datos / "y_train_multiclase.csv"
)

print(
    "Clases del modelo binario:",
    sorted(y_binario_comprobacion["riesgo_binario"].unique())
)

print(
    "Clases del modelo multiclase:",
    sorted(y_multiclase_comprobacion["nivel_riesgo"].unique())
)

Clases del modelo binario: [np.int64(0), np.int64(1)]
Clases del modelo multiclase: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


# 24. Conclusiones del preprocesamiento

El conjunto German Credit fue preparado correctamente para el entrenamiento
de los modelos de clasificación binaria y multiclase.

Las 20 variables originales se conservaron después de revisar su significado,
calidad y utilidad. Luego de aplicar `get_dummies`, estas variables quedaron
representadas mediante 61 columnas numéricas.

Los datos fueron divididos en 800 registros de entrenamiento y 200 registros
de prueba, conservando la distribución original de créditos buenos y malos.

El escalador `MinMaxScaler` fue ajustado únicamente con el conjunto de
entrenamiento. Los datos resultantes se encuentran normalizados y no contienen
valores nulos.

Para el modelo binario se utiliza la siguiente codificación:

- `0`: crédito bueno.
- `1`: crédito malo.

Para el modelo multiclase se utiliza:

- `0`: Bajo.
- `1`: Medio.
- `2`: Alto.
- `3`: Crítico.

Los cuatro niveles corresponden a una clasificación académica construida a
partir de la clase original y de un puntaje de exposición. No representan una
escala bancaria oficial.

Los datos procesados, el escalador, el orden de las columnas y las reglas de
transformación quedaron guardados para que los modelos, la API y la interfaz
utilicen exactamente el mismo procedimiento.

# 25. Implementación modular mediante POO

Después de validar el preprocesamiento paso a paso, la misma lógica fue
trasladada al archivo `src/data_prep.py`.

La clase `PreparadorDatos` organiza las operaciones de:

- Carga y verificación del dataset.
- Transformación de la variable objetivo binaria.
- Conversión de variables categóricas.
- División en entrenamiento y prueba.
- Normalización con `MinMaxScaler`.
- Creación de los cuatro niveles de riesgo.
- Verificación de resultados.
- Guardado de los archivos procesados.

El notebook conserva el procedimiento detallado para facilitar su explicación
y trazabilidad. La implementación mediante POO permite reutilizar exactamente
la misma lógica desde los modelos, la API y la interfaz.


## Ejemplo de uso de la clase

La clase puede ejecutar todo el preprocesamiento mediante la creación de un
objeto y una llamada al método principal.

In [63]:
# Ejemplo de ejecución de la clase.
# Se deja comentado porque los archivos ya fueron generados.

# from src.data_prep import PreparadorDatos
#
# preparador = PreparadorDatos()
# preparador.ejecutar_preprocesamiento()